# Stage 04b — Model Building (XGBoost Feature Importance)

Build a logistic regression PD model using XGBoost feature importance for variable selection.

**Method:** Train XGBoost on WoE-encoded shortlisted variables, rank by gain-based importance, select top variables by cumulative importance threshold, then fit final logistic regression with statsmodels.

**Inputs:**
- Binned dataset: `{RUN_DIR}/data/loans_binned.csv`
- Shortlisted variables from Stage 03
- Target: `Creditability`

In [ ]:
import sys, os
os.chdir('C:/projects/superagent')
sys.path.insert(0, 'src')
import pdtoolkit as pdt
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from xgboost import XGBClassifier
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import json
import warnings
warnings.filterwarnings('ignore')

RUN_DIR = 'runs/2026-03-17_071354'

# Plot settings
BLUE = '#2166AC'
RED = '#D6604D'
GREY = '#999999'
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 150

In [ ]:
# Load binned dataset
db_binned = pd.read_csv(f'{RUN_DIR}/data/loans_binned.csv')
target = 'Creditability'

# Shortlisted variables from Stage 03
shortlist = [
    'Account Balance',
    'Payment Status of Previous Credit',
    'Duration of Credit (month)',
    'Value Savings/Stocks',
    'Purpose',
    'Credit Amount',
    'Most valuable available asset',
    'Age (years)'
]

print(f'Binned dataset shape: {db_binned.shape}')
print(f'Target distribution:\n{db_binned[target].value_counts()}')
print(f'Default rate: {db_binned[target].mean():.4f}')
print(f'Shortlisted variables: {len(shortlist)}')

## WoE Encoding

In [ ]:
# Ensure all shortlisted variables are string type for pdt.replace_woe
db_subset = db_binned[shortlist + [target]].copy()
for col in shortlist:
    db_subset[col] = db_subset[col].astype(str)

# WoE encode
db_woe, woe_mapping = pdt.replace_woe(db_subset, target)

# Get bivariate summary for WoE mapping details (replace_woe mapping may be empty)
bv_summary, bv_info = pdt.bivariate(db_subset, target)

print('WoE-encoded dataset shape:', db_woe.shape)
print()
print('Bivariate WoE mapping (first 20 rows):')
print(bv_summary[['rf', 'bin', 'woe', 'iv_b']].head(20).to_string())

## XGBoost Feature Importance

In [ ]:
# Prepare data for XGBoost
X_woe = db_woe[shortlist].copy()
y = db_woe[target].copy()

# Fit XGBoost on WoE-encoded shortlisted variables
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)
xgb_model.fit(X_woe, y)

# Extract gain-based feature importances
importances = pd.Series(
    xgb_model.feature_importances_,
    index=shortlist
).sort_values(ascending=False)

print('XGBoost Feature Importances (gain-based):')
print(importances.to_string())

In [ ]:
# Variable selection by cumulative importance
cum_importance = importances.cumsum() / importances.sum()

# Build importance table
importance_df = pd.DataFrame({
    'Variable': importances.index,
    'Importance': importances.values,
    'Cumulative_Pct': cum_importance.values
})

# Select variables covering 90% of cumulative importance
CUMULATIVE_THRESHOLD = 0.90
selected_mask = cum_importance <= CUMULATIVE_THRESHOLD
# Always include the variable that crosses the threshold
if selected_mask.sum() < len(selected_mask):
    idx_cross = selected_mask.sum()
    selected_mask.iloc[idx_cross] = True

selected_variables = importances[selected_mask].index.tolist()

# Enforce min 4, max 12
if len(selected_variables) < 4:
    selected_variables = importances.head(4).index.tolist()
elif len(selected_variables) > 12:
    selected_variables = importances.head(12).index.tolist()

importance_df['Selected'] = importance_df['Variable'].isin(selected_variables)

print(f'Cumulative importance threshold: {CUMULATIVE_THRESHOLD}')
print(f'Selected {len(selected_variables)} variables:')
for v in selected_variables:
    print(f'  - {v} (importance: {importances[v]:.4f}, cumulative: {cum_importance[v]:.4f})')

excluded_from_shortlist = [v for v in shortlist if v not in selected_variables]
print(f'\nExcluded from shortlist: {excluded_from_shortlist}')

In [ ]:
# Plot: XGBoost feature importance with cumulative line
fig, ax1 = plt.subplots(figsize=(10, 6))

# Bar chart for importance
colors = [BLUE if v in selected_variables else GREY for v in importance_df['Variable']]
bars = ax1.barh(range(len(importance_df)), importance_df['Importance'].values, color=colors)
ax1.set_yticks(range(len(importance_df)))
ax1.set_yticklabels(importance_df['Variable'].values)
ax1.set_xlabel('Feature Importance (Gain)')
ax1.set_title('XGBoost Feature Importance — Variable Selection for Stage 04b')
ax1.invert_yaxis()

# Cumulative line on secondary axis
ax2 = ax1.twiny()
ax2.plot(importance_df['Cumulative_Pct'].values, range(len(importance_df)),
         color=RED, marker='o', linewidth=2, markersize=6)
ax2.axvline(x=CUMULATIVE_THRESHOLD, color=RED, linestyle='--', alpha=0.5, label=f'{CUMULATIVE_THRESHOLD:.0%} threshold')
ax2.set_xlabel('Cumulative Importance (%)', color=RED)
ax2.tick_params(axis='x', labelcolor=RED)
ax2.legend(loc='lower right')

plt.tight_layout()
plt.savefig(f'{RUN_DIR}/figures/04b_xgb_importance.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: 04b_xgb_importance.png')

## Logistic Regression Model Fit

In [ ]:
# Fit logistic regression with statsmodels on selected WoE variables
X_selected = sm.add_constant(X_woe[selected_variables])
logit_model = sm.Logit(y, X_selected)
logit_res = logit_model.fit(disp=0)

print(logit_res.summary())

In [ ]:
# Extract coefficients table
summary_df = pd.DataFrame({
    'variable': ['const'] + selected_variables,
    'coef': logit_res.params.values,
    'std_err': logit_res.bse.values,
    'z': logit_res.tvalues.values,
    'p_value': logit_res.pvalues.values,
    'ci_lower': logit_res.conf_int()[0].values,
    'ci_upper': logit_res.conf_int()[1].values
})

print('Coefficients Table:')
print(summary_df.to_string(index=False))

# Model-level statistics
print(f'\nModel Statistics:')
print(f'  Log-Likelihood: {logit_res.llf:.3f}')
print(f'  LL-Null: {logit_res.llnull:.3f}')
print(f'  Pseudo R-squared: {logit_res.prsquared:.4f}')
print(f'  LLR p-value: {logit_res.llr_pvalue:.6e}')
print(f'  Df Model: {logit_res.df_model:.0f}')
print(f'  No. Observations: {logit_res.nobs:.0f}')

## Self-Assessment Checks

In [ ]:
# Check 1: Variable count
n_vars = len(selected_variables)
print(f'Check 1 — Variable count: {n_vars} (target: 4-12)')
assert 4 <= n_vars <= 12, f'Variable count {n_vars} outside [4, 12]'
print('  PASS')

# Check 2: Coefficient signs vs WoE directions
# In pdtoolkit's WoE convention: positive WoE = low risk (good), negative WoE = high risk (bad)
# In logit P(default=1|X) = 1/(1+exp(-XB)):
#   Negative coefficient on WoE means: higher WoE (lower risk) -> lower P(default) — CORRECT
# So all WoE coefficients should be NEGATIVE (same sign = consistent direction)
print(f'\nCheck 2 — Coefficient sign consistency:')
coefficients = logit_res.params.drop('const')
signs_ok = True
sign_details = []
for var in selected_variables:
    coef = coefficients[var]
    # WoE coefficients should all be negative in pdtoolkit convention
    # (higher WoE = lower risk, negative coef means higher WoE lowers P(default))
    consistent = coef < 0
    sign_details.append({'variable': var, 'coef': coef, 'consistent': consistent})
    status = 'OK' if consistent else 'REVERSED'
    print(f'  {var}: coef={coef:.4f} — {status}')
    if not consistent:
        signs_ok = False

print(f'  All signs consistent: {signs_ok}')

In [ ]:
# Handle sign reversals if any — swap with next-ranked XGBoost variable
flags = []
iteration = 0
max_iterations = 3

while not signs_ok and iteration < max_iterations:
    iteration += 1
    print(f'\n--- Sign reversal fix iteration {iteration} ---')
    
    # Find reversed variables
    reversed_vars = [d['variable'] for d in sign_details if not d['consistent']]
    
    # Find available replacement variables (from shortlist, not currently selected)
    available = [v for v in importances.index if v not in selected_variables]
    
    for rv in reversed_vars:
        if available:
            replacement = available.pop(0)
            print(f'  Swapping {rv} (reversed) with {replacement}')
            selected_variables.remove(rv)
            selected_variables.append(replacement)
            excluded_from_shortlist = [v for v in shortlist if v not in selected_variables]
        else:
            print(f'  No replacement available for {rv} — removing')
            selected_variables.remove(rv)
            flags.append(f'Removed {rv} due to sign reversal, no replacement available')
    
    # Refit model
    X_selected = sm.add_constant(X_woe[selected_variables])
    logit_model = sm.Logit(y, X_selected)
    logit_res = logit_model.fit(disp=0)
    
    # Recheck signs (negative = consistent in pdtoolkit WoE convention)
    coefficients = logit_res.params.drop('const')
    signs_ok = True
    sign_details = []
    for var in selected_variables:
        coef = coefficients[var]
        consistent = coef < 0
        sign_details.append({'variable': var, 'coef': coef, 'consistent': consistent})
        if not consistent:
            signs_ok = False
    
    print(f'  Signs consistent after iteration {iteration}: {signs_ok}')

if not signs_ok:
    flags.append('Coefficient sign reversal persists after max iterations')
    print('WARNING: Sign reversals remain — flagging for review')
else:
    print('All coefficient signs are consistent.')

# Rebuild coefficients table after possible changes
summary_df = pd.DataFrame({
    'variable': ['const'] + selected_variables,
    'coef': logit_res.params.values,
    'std_err': logit_res.bse.values,
    'z': logit_res.tvalues.values,
    'p_value': logit_res.pvalues.values,
    'ci_lower': logit_res.conf_int()[0].values,
    'ci_upper': logit_res.conf_int()[1].values
})
print('\nFinal Coefficients:')
print(summary_df.to_string(index=False))

In [ ]:
# Check 3: VIF (Variance Inflation Factor)
print('Check 3 — Multicollinearity (VIF):')
X_vif = X_woe[selected_variables].copy()
vif_data = []
for i, var in enumerate(selected_variables):
    vif_val = variance_inflation_factor(X_vif.values, i)
    vif_data.append({'variable': var, 'vif': vif_val})
    status = 'OK' if vif_val < 5 else ('WARNING' if vif_val < 10 else 'HIGH')
    print(f'  {var}: VIF={vif_val:.2f} — {status}')

vif_ok = all(d['vif'] < 10 for d in vif_data)
if not vif_ok:
    flags.append('High VIF detected (>10) — multicollinearity concern')
    print('  WARNING: High VIF detected')
else:
    print('  All VIF values < 10 — PASS')

In [ ]:
# Model performance metrics
y_pred = logit_res.predict(X_selected)
model_auc = pdt.auc_model(y_pred.values, y.values)
model_gini = 2 * model_auc - 1

# KS statistic
fpr, tpr, thresholds = roc_curve(y, y_pred)
model_ks = max(tpr - fpr)

print(f'Model Performance:')
print(f'  AUC:  {model_auc:.4f}')
print(f'  Gini: {model_gini:.4f}')
print(f'  KS:   {model_ks:.4f}')

# Check 3 (performance): Minimum discrimination
if model_auc < 0.60:
    flags.append(f'AUC {model_auc:.4f} below minimum threshold 0.60')
    print('  AUC CHECK: FAIL (< 0.60)')
elif model_auc < 0.70:
    flags.append(f'AUC {model_auc:.4f} in weak range (0.60-0.70) — flagged for review')
    print('  AUC CHECK: WARN (0.60-0.70, weak)')
else:
    print('  AUC CHECK: PASS (>= 0.70, acceptable)')

In [ ]:
# Check 4: Score distribution
scores = pdt.scaled_score(y_pred.values, score=600, odd=50, pdo=20)

score_min = scores.min()
score_max = scores.max()
score_mean = scores.mean()
score_std = scores.std()
pct_below_400 = (scores < 400).mean() * 100
pct_above_800 = (scores > 800).mean() * 100

print(f'Check 4 — Score Distribution:')
print(f'  Min:  {score_min:.1f}')
print(f'  Max:  {score_max:.1f}')
print(f'  Mean: {score_mean:.1f}')
print(f'  Std:  {score_std:.1f}')
print(f'  % below 400: {pct_below_400:.2f}%')
print(f'  % above 800: {pct_above_800:.2f}%')

if pct_below_400 > 5:
    flags.append(f'{pct_below_400:.1f}% of scores below 400')
if pct_above_800 > 5:
    flags.append(f'{pct_above_800:.1f}% of scores above 800')

In [ ]:
# Check 5: Decile monotonicity
print('Check 5 — Decile Monotonicity:')
score_df = pd.DataFrame({'score': scores, 'default': y.values})
score_df['decile'] = pd.qcut(score_df['score'], 10, labels=False, duplicates='drop')
decile_stats = score_df.groupby('decile').agg(
    count=('default', 'count'),
    n_defaults=('default', 'sum'),
    default_rate=('default', 'mean'),
    avg_score=('score', 'mean')
).reset_index()

print(decile_stats.to_string(index=False))

# Check monotonicity: default rate should decrease as score increases (higher decile = higher score = lower risk)
dr_values = decile_stats['default_rate'].values
monotonic_decreasing = all(dr_values[i] >= dr_values[i+1] for i in range(len(dr_values)-1))
# Allow near-monotonic (one violation)
violations = sum(1 for i in range(len(dr_values)-1) if dr_values[i] < dr_values[i+1])
decile_monotonic = violations <= 1

print(f'\nStrictly monotonic: {monotonic_decreasing}')
print(f'Monotonicity violations: {violations}')
print(f'Decile monotonicity check: {"PASS" if decile_monotonic else "FAIL"}')

if not decile_monotonic:
    flags.append(f'Decile monotonicity failed with {violations} violations')

In [ ]:
# Check 6: Cross-validation stability
print('Check 6 — Cross-validation Stability:')

# K-fold validation
kfold_result = pdt.kfold_vld(logit_res, db_woe, target, selected_variables)
kfold_auc = kfold_result.summary['auc'].values[0]
print(f'K-fold validation AUC: {kfold_auc:.4f}')
print(kfold_result.iter.to_string(index=False))

# Bootstrap validation
boots_result = pdt.boots_vld(logit_res, db_woe, target, selected_variables)
boots_auc = boots_result.summary['auc'].values[0]
print(f'\nBootstrap validation AUC: {boots_auc:.4f}')
print(boots_result.summary.to_string(index=False))

# Check stability: validation AUC within 0.03 of development AUC
auc_diff_kfold = abs(model_auc - kfold_auc)
auc_diff_boots = abs(model_auc - boots_auc)
print(f'\nDev AUC: {model_auc:.4f}')
print(f'K-fold AUC diff: {auc_diff_kfold:.4f} (threshold: 0.03)')
print(f'Bootstrap AUC diff: {auc_diff_boots:.4f} (threshold: 0.03)')

cv_stable = auc_diff_kfold <= 0.03 and auc_diff_boots <= 0.03
print(f'Cross-validation stability: {"PASS" if cv_stable else "WARN"}')
if not cv_stable:
    flags.append(f'Cross-validation instability: kfold AUC diff={auc_diff_kfold:.4f}, boots AUC diff={auc_diff_boots:.4f}')

## Plots

In [ ]:
# ROC Curve
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(fpr, tpr, color=BLUE, linewidth=2, label=f'XGB Selection Model (AUC={model_auc:.4f})')
ax.plot([0, 1], [0, 1], color=GREY, linestyle='--', label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Stage 04b (XGBoost Selection)')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.savefig(f'{RUN_DIR}/figures/04b_roc_curve.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: 04b_roc_curve.png')

In [ ]:
# Score Distribution
fig, ax = plt.subplots(figsize=(10, 6))
defaults = scores[y.values == 1]
non_defaults = scores[y.values == 0]
ax.hist(non_defaults, bins=30, alpha=0.6, color=BLUE, label='Non-default', density=True)
ax.hist(defaults, bins=30, alpha=0.6, color=RED, label='Default', density=True)
ax.axvline(x=score_mean, color='black', linestyle='--', label=f'Mean={score_mean:.0f}')
ax.set_xlabel('Score')
ax.set_ylabel('Density')
ax.set_title('Score Distribution — Stage 04b (XGBoost Selection)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.savefig(f'{RUN_DIR}/figures/04b_score_distribution.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: 04b_score_distribution.png')

In [ ]:
# Coefficient Plot
fig, ax = plt.subplots(figsize=(10, 6))
coef_vals = logit_res.params.drop('const')
coef_ci = logit_res.conf_int().drop('const')
y_pos = range(len(coef_vals))

colors = [BLUE if c > 0 else RED for c in coef_vals.values]
ax.barh(y_pos, coef_vals.values, color=colors, alpha=0.7)
ax.errorbar(coef_vals.values, y_pos,
            xerr=[coef_vals.values - coef_ci[0].values, coef_ci[1].values - coef_vals.values],
            fmt='none', color='black', capsize=3)
ax.set_yticks(y_pos)
ax.set_yticklabels(coef_vals.index)
ax.axvline(x=0, color='black', linewidth=0.5)
ax.set_xlabel('Coefficient')
ax.set_title('Logistic Regression Coefficients — Stage 04b (XGBoost Selection)')
ax.grid(True, alpha=0.3)
plt.savefig(f'{RUN_DIR}/figures/04b_coefficient_plot.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: 04b_coefficient_plot.png')

In [ ]:
# Score Decile Table (as image)
fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off')
table_data = []
for _, row in decile_stats.iterrows():
    table_data.append([
        int(row['decile']),
        int(row['count']),
        int(row['n_defaults']),
        f"{row['default_rate']:.4f}",
        f"{row['avg_score']:.1f}"
    ])

table = ax.table(
    cellText=table_data,
    colLabels=['Decile', 'Count', 'Defaults', 'Default Rate', 'Avg Score'],
    loc='center',
    cellLoc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 1.4)
ax.set_title('Score Decile Analysis — Stage 04b (XGBoost Selection)', fontsize=12, pad=20)
plt.savefig(f'{RUN_DIR}/figures/04b_score_decile_table.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: 04b_score_decile_table.png')

## Save Model Parameters

In [ ]:
# Build WoE mappings from the bivariate summary DataFrame
woe_mappings_dict = {}
for var in selected_variables:
    var_mapping = bv_summary[bv_summary['rf'] == var][['bin', 'woe']].copy()
    woe_mappings_dict[var] = [
        {'bin': str(row['bin']), 'woe': round(float(row['woe']), 6)}
        for _, row in var_mapping.iterrows()
    ]

# XGBoost importances for all shortlisted variables
xgb_importances_dict = {var: round(float(importances[var]), 6) for var in importances.index}

# Coefficients
coef_dict = {var: round(float(logit_res.params[var]), 6) for var in selected_variables}

model_params = {
    'selection_method': 'xgboost_importance',
    'selected_variables': selected_variables,
    'xgb_importances': xgb_importances_dict,
    'cumulative_importance_threshold': CUMULATIVE_THRESHOLD,
    'woe_mappings': woe_mappings_dict,
    'coefficients': coef_dict,
    'intercept': round(float(logit_res.params['const']), 6),
    'score_params': {
        'base_score': 600,
        'base_odds': 50,
        'pdo': 20
    },
    'model_auc': round(float(model_auc), 4),
    'model_gini': round(float(model_gini), 4),
    'model_ks': round(float(model_ks), 4)
}

with open(f'{RUN_DIR}/pipeline/model_params_xgb.json', 'w') as f:
    json.dump(model_params, f, indent=2)

print(f'Model parameters saved to {RUN_DIR}/pipeline/model_params_xgb.json')
print(json.dumps(model_params, indent=2))

## Stage Summary

| Item | Value | Status |
|---|---|---|
| Selection method | XGBoost importance | - |
| Variables selected | See above | - |
| AUC | See output | PASS if >= 0.70 |
| Gini | See output | PASS if >= 0.35 |
| KS | See output | PASS if >= 0.30 |
| Coefficient signs | See output | PASS/FAIL |
| VIF | All < 10 | PASS/FAIL |
| Score range | See output | PASS if overlaps [400, 800] |
| Decile monotonicity | See output | PASS/FAIL |

**Flags for human review:** See output above

**Recommended action for next stage:** Proceed to model comparison (Stage 04x)